In [1]:
from transformers import T5Tokenizer, T5TokenizerFast, T5ForConditionalGeneration, Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import accuracy_score
import torch
from datasets import Dataset
import evaluate
import pandas as pd
import gc
import numpy as np
import json
import nltk

In [2]:
dataset_path= 'Model_dataset/synthetic_question_ans_data-v2.csv'
cv_path= "Model_dataset/cv.json"
with open(cv_path, "r") as file:
    cv_data= json.load(file)


#qa model
qa_type_model_name= 't5-small'
qa_type_model_result= '.temp/model_results/fine_tuned_question_answer_model-small'
qa_type_model= '.temp/model/fine_tuned_question_answer_model-small'



In [ ]:
cv_data.keys()

### Preprocessing

In [ ]:
df= pd.read_csv(dataset_path)
df= df[["question", "question_type", "answer"]]
df["answer"]= df["answer"].fillna("")
df.info()

In [5]:
df["context"] = df["question_type"].map(cv_data)

In [ ]:
df= df.sample(frac=1).reset_index(drop=True)
df.head()

In [ ]:
df.info()

### Retraing Preparations:

In [8]:
# Preprocess data
def preprocess_data(row):
    input_text = f"question: {row['question']} context: {row['context']}"
    target_text = row['answer']
    return {"input_text": input_text, "target_text": target_text}

processed_data = df.apply(preprocess_data, axis=1)
dataset = Dataset.from_pandas(pd.DataFrame(processed_data.tolist()))

In [9]:
# Split data into train and test
train_test_split = dataset.train_test_split(test_size=0.1)
train_dataset = train_test_split["train"]
test_dataset = train_test_split["test"]

In [ ]:
tokenizer = T5Tokenizer.from_pretrained(qa_type_model_name)

def tokenize_data(example):
    input_encodings = tokenizer(example["input_text"], truncation=True, padding="max_length", max_length=512)
    target_encodings = tokenizer(example["target_text"], truncation=True, padding="max_length", max_length=128)
    input_encodings["labels"] = target_encodings["input_ids"]
    return input_encodings

train_dataset = train_dataset.map(tokenize_data, batched=True)
test_dataset = test_dataset.map(tokenize_data, batched=True)

In [ ]:
nltk.download('punkt')

# Load metrics
rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")

def combine_compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    # Replace -100 in the labels as we can't decode them directly
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Preprocess for BLEU (expects list of tokens)
    bleu_preds = [pred.split() for pred in decoded_preds]
    bleu_labels = [[label.split()] for label in decoded_labels]  # BLEU expects a list of references

    # Compute ROUGE
    rouge_results = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    rouge_results = {key: value.mid.fmeasure * 100 for key, value in rouge_results.items()}
    
    # Compute BLEU
    bleu_result = bleu_metric.compute(predictions=bleu_preds, references=bleu_labels)
    bleu_score = bleu_result["bleu"] * 100  # Convert BLEU to percentage for consistency

    # Combine results
    combined_results = {
        **rouge_results,
        "bleu": bleu_score
    }
    return combined_results


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    # Replace -100 in the labels as we can't decode them directly
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Compute ROUGE
    rouge_results = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    rouge_results = {key: value.mid.fmeasure * 100 for key, value in rouge_results.items()}

    return rouge_results

# Train

In [12]:
model = T5ForConditionalGeneration.from_pretrained(qa_type_model_name)

In [13]:
training_args = TrainingArguments(
    output_dir=qa_type_model_result,           # Output directory
    eval_strategy="epoch",                     # Evaluate every epoch
    save_strategy="epoch",                     # Save every epoch
    learning_rate=3e-5,                        # Learning rate
    num_train_epochs=50,                       # Number of training epochs
    per_device_train_batch_size=4,             # Batch size during training
    per_device_eval_batch_size=4,              # Batch size during evaluation
    gradient_accumulation_steps=2,             # Gradient accumulation steps
    logging_dir="./logs",                      # Directory for logs
    logging_steps=10,                          # Log every 10 steps
    save_total_limit=4,                        # Limit the number of saved checkpoints
    warmup_steps=800,                          # Warmup steps for learning rate
    weight_decay=0.01,                         # Weight decay for regularization
    adam_epsilon=1e-8,                        # Epsilon for the Adam optimizer
    max_grad_norm=1.0,                        # Max gradient norm for gradient clipping
    # fp16=True,                               # Enable mixed precision training (optional)
    use_cpu=True,                            # Force CPU usage (if necessary)
    load_best_model_at_end=True,               # Load best model at the end of training
    metric_for_best_model="rougeL",         # Metric to monitor for best model
    greater_is_better=True,                    # Set to True for accuracy metrics
)

In [ ]:
# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)


# Clearing memory before training
torch.cuda.empty_cache()
gc.collect()

# Train the model
trainer.train()

In [ ]:
evaluation_results = trainer.evaluate()
evaluation_results

In [ ]:
trainer.save_model(qa_type_model)
tokenizer.save_pretrained(qa_type_model)